# Segmento 1: Embeddings e Ricerca Semantica da Zero

Abbiamo 13.500 ricette. La ricerca per keyword funziona per match esatti, ma fallisce quando le parole dell'utente non corrispondono a quelle della ricetta.

Trasformiamo il testo in **vettori** e cerchiamo per **significato**.

In [2]:
from dotenv import load_dotenv
from openai import OpenAI
from datasets import load_dataset
import numpy as np
import pandas as pd
import ast, os

load_dotenv()
client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"

ds = load_dataset("Hieu-Pham/kaggle_food_recipes", split="train")
df = ds.to_pandas().drop(columns=["Unnamed: 0", "Image_Name"])

# Prendiamo un sottoinsieme di 1000 ricette
df = df.sample(n=1000, random_state=42).reset_index(drop=True)
print(f"{len(df)} ricette caricate")

1000 ricette caricate


## Preparare il testo per l'embedding

Concateniamo **titolo + ingredienti + istruzioni** in un unico testo per ogni ricetta. Così la ricerca semantica può fare match su qualsiasi parte della ricetta.

In [3]:
def safe_str(val):
    """Convert to string, handling NaN/None."""
    if pd.isna(val):
        return ""
    return str(val)

def recipe_to_text(row):
    """Concatenate all recipe fields into a single string for embedding."""
    title = safe_str(row["Title"])
    ingredients = safe_str(row["Ingredients"])
    instructions = safe_str(row["Instructions"])
    return f"Title: {title}\nIngredients: {ingredients}\nInstructions: {instructions}"

df["full_text"] = df.apply(recipe_to_text, axis=1)

# Vediamo com'è fatto
print(df["full_text"].iloc[0][:500])

Title: Hazelnut Cookies
Ingredients: ['1/2 cup hazelnuts (2 oz)', '1/4 cup plus 3 tablespoons sugar', '3/4 cup plus 2 tablespoons all-purpose flour', '1 stick (1/2 cup) cold unsalted butter, cut into small pieces']
Instructions: Put oven rack in middle position and preheat oven to 350°F.
Toast hazelnuts in a shallow baking pan until fragrant and skins begin to loosen, about 6 minutes. Rub nuts in a kitchen towel to remove any loose skins (some skins may not come off) and cool to room temperature


## Generare gli embeddings con OpenAI

Mandiamo il testo di ogni ricetta al modello `text-embedding-3-small` di OpenAI e riceviamo indietro un vettore (una lista di 1536 numeri).

In [14]:
def get_embeddings(texts, model=EMBED_MODEL):
    """Get embeddings for a list of texts using OpenAI."""
    response = client.embeddings.create(input=texts, model=model)
    return [item.embedding for item in response.data]

# Embeddiamo in batch da 100 per evitare rate limit
all_embeddings = []
batch_size = 100

for i in range(0, len(df), batch_size):
    batch = df["full_text"].iloc[i:i+batch_size].tolist()
    embeddings = get_embeddings(batch)
    all_embeddings.extend(embeddings)
    print(f"Embedding: {min(i+batch_size, len(df))}/{len(df)} ricette")

# Convertiamo in una matrice NumPy
embedding_matrix = np.array(all_embeddings)
print(f"\nDimensione matrice: {embedding_matrix.shape}")
print(f"Ogni ricetta è ora un vettore di {embedding_matrix.shape[1]} numeri")

Embedding: 100/1000 ricette
Embedding: 200/1000 ricette
Embedding: 300/1000 ricette
Embedding: 400/1000 ricette
Embedding: 500/1000 ricette
Embedding: 600/1000 ricette
Embedding: 700/1000 ricette
Embedding: 800/1000 ricette
Embedding: 900/1000 ricette
Embedding: 1000/1000 ricette

Dimensione matrice: (1000, 1536)
Ogni ricetta è ora un vettore di 1536 numeri


## Cosine similarity, a mano

Due vettori sono simili se puntano nella stessa direzione. La **cosine similarity** misura esattamente questo: il coseno dell'angolo tra di essi.

- 1.0 = stessa direzione (stesso significato)
- 0.0 = perpendicolari (non correlati)
- -1.0 = direzione opposta (raro con gli embeddings)

In [6]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def search_recipes(query, top_n=5):
    """Search recipes by semantic similarity to a query string."""
    query_embedding = get_embeddings([query])[0]
    query_vec = np.array(query_embedding)
    
    # Calcoliamo la similarità con ogni ricetta
    similarities = np.array([
        cosine_similarity(query_vec, embedding_matrix[i])
        for i in range(len(embedding_matrix))
    ])
    
    # Prendiamo i top N indici
    top_indices = similarities.argsort()[::-1][:top_n]
    
    results = []
    for idx in top_indices:
        results.append({
            "title": df.iloc[idx]["Title"],
            "similarity": round(similarities[idx], 4),
        })
    return results

# Test rapido
for r in search_recipes("chicken with garlic"):
    print(f"  {r['similarity']:.4f}  {r['title']}")

  0.5325  Chicken Skin With Peanuts, Chiles, and Lime
  0.5264  Pan-Seared Chicken with Tarragon Butter Sauce
  0.5113  Crispy Chicken Breasts with Chermoula and Escarole
  0.5090  Herb-Rubbed Cast-Iron Chicken with Pan Sauce
  0.5066  Sicilian Grill-Roasted Chicken


## Esempio 1: "marinara" vs "tomato"

La **ricerca per keyword** su "marinara" non troverebbe mai una ricetta che dice solo "tomato sauce" senza usare la parola "marinara".

La **ricerca semantica** sa che marinara = salsa a base di pomodoro.

In [7]:
# Ricerca per keyword: trova solo ricette con la parola letterale "marinara"
keyword_hits = df[df["full_text"].str.contains("marinara", case=False)]
keyword_miss = df[~df["full_text"].str.contains("marinara", case=False)]

print(f"Keyword 'marinara': {len(keyword_hits)} trovate, {len(keyword_miss)} perse\n")

# Ricerca semantica: trova ricette per significato
print("Ricerca semantica per 'dish with marinara':")
for r in search_recipes("dish with marinara", top_n=10):
    # Verifichiamo se la ricetta contiene davvero la parola "tomato"
    row = df[df["Title"] == r["title"]].iloc[0]
    has_tomato_keyword = "marinara" in row["full_text"].lower()
    marker = "" if has_tomato_keyword else " <-- niente keyword 'marinara'!"
    print(f"  {r['similarity']:.4f}  {r['title']}{marker}")

Keyword 'marinara': 5 trovate, 995 perse

Ricerca semantica per 'dish with marinara':
  0.5182  Antipasto Pasta <-- niente keyword 'marinara'!
  0.5026  Mozzarella Arrabiata Salsa <-- niente keyword 'marinara'!
  0.4997  Rigatoni with Shrimp, Calamari and Basil <-- niente keyword 'marinara'!
  0.4996  Anchovies in Tomato Sauce with Pasta <-- niente keyword 'marinara'!
  0.4928  Pasta alla Norma <-- niente keyword 'marinara'!
  0.4871  Spaghetti with Olive and Pine Nut Salsa <-- niente keyword 'marinara'!
  0.4817  Pasta With 15-Minute Meat Sauce <-- niente keyword 'marinara'!
  0.4769  Pasta with Spicy Sausage, Radicchio, and Sun-Dried Tomatoes <-- niente keyword 'marinara'!
  0.4725  Jumbo Shrimp Marsala Housewife-Style (Gamberoni alla Casalinga Siciliana) <-- niente keyword 'marinara'!
  0.4722  Pantry Pasta Puttanesca <-- niente keyword 'marinara'!


In [10]:
def print_recipe(title):
    """Print full recipe text for a given title."""
    match = df[df["Title"] == title]
    if match.empty:
        print(f"Ricetta '{title}' non trovata.")
        return
    print(match.iloc[0]["full_text"])

print_recipe("Mozzarella Arrabiata Salsa")

Title: Mozzarella Arrabiata Salsa
Ingredients: ['2 pounds tomatoes, divided', '1 fresh hot cherry pepper (optional), stemmed and chopped', '1/4 teaspoon hot red pepper flakes', '6 tablespoons extra-virgin olive oil', '1 celery rib, finely chopped', '1 pound fresh mozzarella, chopped', '2 tablespoons chopped celery leaves', 'Accompaniment: 1 pound capellini, cooked and drained']
Instructions: Halve 1 pound tomatoes, then purée with cherry pepper (if using), red pepper flakes, oil, and 1/2 teaspoon salt in a blender until smooth.
Finely chop remaining tomatoes and combine with celery and tomato purée in a large bowl.
Toss hot capellini with tomato sauce and mozzarella. Serve sprinkled with celery leaves.


## Esempio 2: "quick weeknight dinner"

Nessuna ricetta nel dataset contiene le parole "quick weeknight dinner". La ricerca per keyword restituisce **zero risultati**.

La ricerca semantica capisce l'**intento**: tempo di preparazione breve, tecniche semplici, ingredienti di tutti i giorni.

In [11]:
# Ricerca per keyword: zero risultati
keyword_hits = df[df["full_text"].str.contains("quick weeknight dinner", case=False)]
print(f"Keyword 'quick weeknight dinner': {len(keyword_hits)} risultati\n")

# Ricerca semantica: capisce l'intento
print("Ricerca semantica per 'quick weeknight dinner':")
for r in search_recipes("quick weeknight dinner"):
    print(f"  {r['similarity']:.4f}  {r['title']}")

Keyword 'quick weeknight dinner': 0 risultati

Ricerca semantica per 'quick weeknight dinner':
  0.4669  Pasta With 15-Minute Meat Sauce
  0.4578  Mac 'n' Cheese Minis
  0.4412  Quick Coq au Vin
  0.4384  Seafood Stew for Two
  0.4371  Creamy Farfalle with Salmon and Peas


## Esempio 3: "lemon juice" vs "juice of 2 lemons"

Un match esatto per la frase `"lemon juice"` non trova ricette che scrivono `"juice of 2 lemons"` o `"freshly squeezed lemon"`.

La ricerca semantica non si preoccupa dell'ordine delle parole, capisce il concetto.

In [13]:
# Ricerca per keyword: match esatto della frase
exact_match = df[df["full_text"].str.contains("lemon juice", case=False)]
alt_match = df[df["full_text"].str.contains("juice of", case=False) & df["full_text"].str.contains("lemon", case=False)]
only_alt = alt_match[~alt_match.index.isin(exact_match.index)]

print(f"Frase esatta 'lemon juice': {len(exact_match)} trovate")
print(f"'juice of ... lemon' (formulazione diversa): {len(only_alt)} ricette aggiuntive perse dalla keyword\n")

# Ricerca semantica: le trova tutte
print("Ricerca semantica per 'recipe with lemon juice':")
for r in search_recipes("recipe with lemon juice. Ingredients: lemon. Instructions: squeeze the lemon"):
    print(f"  {r['similarity']:.4f}  {r['title']}")

Frase esatta 'lemon juice': 160 trovate
'juice of ... lemon' (formulazione diversa): 6 ricette aggiuntive perse dalla keyword

Ricerca semantica per 'recipe with lemon juice':
  0.5006  Preserved Lemons
  0.5002  Lemon Curd with Berries
  0.4609  Lemon-Chicken Drumsticks
  0.4603  Lamb with Preserved Lemons
  0.4578  Lemon Curd Tart with Olive Oil


## Cosa abbiamo visto

La cosine similarity con NumPy ci dà la **ricerca semantica**, ma confrontiamo ogni vettore, ogni volta.

Prossimo passo: mettiamo questi vettori in un **vector database** che gestisca tutto in modo efficiente.